# Homework 01: Binary Classification with Neural Networks (Reference Solution)

This notebook is a **fully completed reference implementation** for Homework 01.
It is intended for **comparison and learning**, not submission.

Each problem includes:
- Clear Markdown explaining *what the professor is teaching*
- Well-documented, production-style code
- Humanized interpretation at a Master's level


## Load and Inspect the Data

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

df.head()

## Problem 1A: Class Imbalance

We first inspect the target distribution to understand whether class imbalance
may bias training.


In [ ]:
import numpy as np
from collections import Counter

X = df.drop(columns='target').values
y = df['target'].values

print("X shape:", X.shape)
print("y shape:", y.shape)

Counter(y)

In [ ]:
# Proportion of class 1
a1a = Counter(y)[1] / len(y)
a1a

**Explanation:**  
The dataset is imbalanced, with more benign (1) than malignant (0) cases.
This motivates stratification and class weighting.


## Problem 1B: Stratified Split and Standardization

Stratification ensures that train and test sets preserve class proportions.
Standardization is performed *after* splitting to avoid data leakage.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

a1b = Counter(y_train)[1] / len(y_train)
a1b

## Problem 1C: Baseline Logistic Regression Network

A single-neuron sigmoid network is equivalent to logistic regression.
We apply **class weighting** so minority-class errors matter more.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

def compute_class_weights(y):
    counts = Counter(y)
    n = len(y)
    k = len(counts)
    return {cls: n / (k * cnt) for cls, cnt in counts.items()}

weights = compute_class_weights(y_train)

baseline_model = Sequential([
    Dense(1, activation='sigmoid', input_shape=(X_train.shape[1],))
])

baseline_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

baseline_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    class_weight=weights
)

## Problem 1D: Baseline Model Testing

We evaluate performance on unseen data to measure generalization.


In [ ]:
test_loss, test_accuracy = baseline_model.evaluate(X_test, y_test)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

---
## Reusable Training Function

Defining a reusable function reflects strong development practice
and keeps experiments consistent.


In [ ]:
def train_and_evaluate(hidden_layers):
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(units, activation='sigmoid', input_shape=(X_train.shape[1],)))
        else:
            model.add(Dense(units, activation='sigmoid'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        class_weight=weights
    )

    return model.evaluate(X_test, y_test)

## Problem 2: One Hidden Layer (16 units)

In [ ]:
loss, acc = train_and_evaluate([16])
loss, acc

## Problem 3: One Hidden Layer (64 units)

In [ ]:
loss, acc = train_and_evaluate([64])
loss, acc

## Problem 4: One Hidden Layer (256 units)

In [ ]:
loss, acc = train_and_evaluate([256])
loss, acc

## Problem 5: Two Hidden Layers (64, 32 units)

In [ ]:
loss, acc = train_and_evaluate([64, 32])
loss, acc